# Практика: lambda-функции и паттерн «Стратегия»

На этом занятии обсудим:

- анонимные функции и выражение `lambda`;
- использование `lambda` в параметре `key` функций `sorted`, `min` и `max`;
- паттерн проектирования «Стратегия» и его реализацию на обычных функциях;

## 1. Анонимные lambda-функции

### Назначение

Часто функция требуется ровно для одного применения: ее передают в качестве
аргумента (например, в параметр `key`) и больше нигде не используют.
Определять для этого именованную функцию через `def` избыточно: нужно
придумать имя и вынести короткое вычисление в отдельный блок кода.

**Анонимная функция** — объект-функция, созданный без привязки к имени. В
Python анонимные функции создаются **выражением `lambda`**:

```python
lambda параметры: выражение
```

- `lambda` — ключевое слово;
- параметры перечисляются через запятую так же, как в `def`, но без скобок;
- после двоеточия записывается **одно выражение**. Его значение вычисляется
  при каждом вызове функции и становится возвращаемым значением; слова
  `return` в теле нет.

Выражение `lambda` возвращает объект-функцию, поэтому его можно записать в
любом месте, где допустимо выражение: в аргументе вызова, в элементе списка,
в значении словаря.

In [ ]:
def square(number: int) -> int:
    return number * number


square_lambda = lambda number: number * number

print(square(4))            # 16
print(square_lambda(4))     # 16
print(type(square_lambda))  # <class 'function'>

Функция, созданная через `lambda`, не отличается от функции, созданной через
`def`: у нее тот же тип `function`, она вызывается так же и может быть
передана в другую функцию. Единственное заметное отличие — имя: у анонимной
функции атрибут `__name__` всегда равен `<lambda>`.

In [ ]:
print(square.__name__)         # square
print(square_lambda.__name__)  # <lambda>

Присваивание lambda-функции переменной (`square_lambda = lambda ...`) в
примере выше выполнено только для сравнения с `def`. Руководство PEP 8
рекомендует в таком случае использовать `def`: у функции появляется
осмысленное имя, которое отображается, например, в трассировке ошибки.
Анонимная функция нужна там, где имя не требуется.

### Параметры lambda-функций

Параметры lambda-функции задаются по тем же правилам, что и у обычной:
допустимы позиционные параметры, значения по умолчанию, `*args` и
`**kwargs`. Чтобы вызвать анонимную функцию сразу после создания, выражение
`lambda` заключают в круглые скобки:

In [ ]:
print((lambda: "no parameters")())                                       # no parameters
print((lambda first_term, second_term: first_term + second_term)(2, 3))  # 5
print((lambda number, power=2: number ** power)(5))                      # 25
print((lambda *numbers: sum(numbers))(1, 2, 3))                          # 6
print((lambda **options: sorted(options))(second=1, first=2))            # ['first', 'second']
print((lambda number: "even" if number % 2 == 0 else "odd")(7))          # odd

Как видно из последней строки, в теле lambda-функции можно использовать
условное выражение `... if ... else ...`. Это именно выражение, а не
инструкция `if`.

### Ограничения lambda-функций

- Тело состоит из **одного выражения**. Инструкции (`return`, присваивание,
  `for`, `while`, `if` как инструкция, `try`, `raise`, `assert`) в нем
  недопустимы.
- Нельзя указать аннотации типов параметров и возвращаемого значения.
- Нельзя задать docstring.
- Сложное выражение внутри lambda трудно читать и отлаживать.

In [ ]:
lambda number: return number * 2

При использовании инструкции внутри lambda возникает `SyntaxError`. Если
логика не помещается в одно простое выражение, следует определить функцию
через `def`.

### Обращение к внешним именам

Имена, которых нет в локальном пространстве имен lambda-функции, ищутся во
внешних областях видимости по тем же правилам, что и в обычной функции.
Поиск выполняется **в момент вызова**, а не в момент создания функции:

In [ ]:
discount_rate = 0.1
apply_discount = lambda price: price * (1 - discount_rate)

print(apply_discount(100.0))  # 90.0

discount_rate = 0.5

print(apply_discount(100.0))  # 50.0

### lambda как параметр `key`

В конспекте о функциях как объектах мы выяснили, что параметр `key` функций
`sorted`, `min`, `max` и метода `list.sort` принимает функцию одного
аргумента. Она вызывается один раз для каждого элемента коллекции, а
сравниваются уже значения, которые она вернула. Это типичная область
применения анонимных функций: ключ — короткое выражение, нужное только в
одном вызове.

#### Функция `sorted`

Сортировка слов по длине и по последней букве:

In [ ]:
words = ["banana", "kiwi", "apple", "fig", "cherry"]

print(sorted(words, key=lambda word: len(word)))               # ['fig', 'kiwi', 'apple', 'banana', 'cherry']
print(sorted(words, key=lambda word: word[-1]))                # ['banana', 'apple', 'fig', 'kiwi', 'cherry']
print(sorted(words, key=lambda word: word[-1], reverse=True))  # ['cherry', 'kiwi', 'fig', 'apple', 'banana']

Если в качестве ключа подходит готовая функция, lambda-обертка не нужна:
`key=lambda word: len(word)` эквивалентно `key=len`, но записывается длиннее.

In [ ]:
print(sorted(words, key=len))  # ['fig', 'kiwi', 'apple', 'banana', 'cherry']

Элементы коллекции часто составные (кортежи, списки, словари). Анонимная
функция позволяет выбрать поле, по которому выполняется сортировка:

In [ ]:
people = [("Clara", 25), ("Boris", 19), ("Anna", 25), ("Dmitry", 31)]

print(sorted(people, key=lambda person: person[1]))
# [('Boris', 19), ('Clara', 25), ('Anna', 25), ('Dmitry', 31)]

print(sorted(people, key=lambda person: person[0]))
# [('Anna', 25), ('Boris', 19), ('Clara', 25), ('Dmitry', 31)]

Сортировка устойчивая, поэтому `Clara` и `Anna`, у которых одинаковый ключ
`25`, остаются в исходном порядке. Чтобы при равном возрасте упорядочить людей
по имени, ключом делают **кортеж**. Если по одному полю нужен порядок
убывания, а по другому — возрастания, числовое поле берется со знаком минус
(параметр `reverse=True` развернул бы порядок сразу по всем полям ключа):

In [ ]:
print(sorted(people, key=lambda person: (person[1], person[0])))
# [('Boris', 19), ('Anna', 25), ('Clara', 25), ('Dmitry', 31)]

print(sorted(people, key=lambda person: (-person[1], person[0])))
# [('Dmitry', 31), ('Anna', 25), ('Clara', 25), ('Boris', 19)]

Для словарей ключ выбирает значение по ключу словаря или вычисляет выражение
по нескольким полям:

In [ ]:
products = [
    {"label": "laptop", "price": 1200.0, "amount": 1},
    {"label": "mouse", "price": 25.0, "amount": 4},
    {"label": "monitor", "price": 300.0, "amount": 5},
    {"label": "cable", "price": 5.0, "amount": 10},
]

by_price = sorted(products, key=lambda product: product["price"])
print([product["label"] for product in by_price])  # ['cable', 'mouse', 'monitor', 'laptop']

by_cost = sorted(
    products,
    key=lambda product: product["price"] * product["amount"],
    reverse=True,
)
print([product["label"] for product in by_cost])  # ['monitor', 'laptop', 'mouse', 'cable']

Сортировка без учета регистра и сортировка пар `(ключ, значение)` словаря по
значению:

In [ ]:
names = ["banana", "Cherry", "apple"]

print(sorted(names))                                 # ['Cherry', 'apple', 'banana']
print(sorted(names, key=lambda name: name.lower()))  # ['apple', 'banana', 'Cherry']
print(sorted(names, key=str.lower))                  # ['apple', 'banana', 'Cherry'] — lambda не нужна

scores = {"Anna": 82, "Boris": 95, "Clara": 78}

print(sorted(scores.items(), key=lambda pair: pair[1], reverse=True))
# [('Boris', 95), ('Anna', 82), ('Clara', 78)]

#### Функции `max` и `min`

Функции `max` и `min` тоже принимают параметр `key`. Они возвращают
**элемент**, у которого ключ наибольший (наименьший), а не значение ключа.
Если таких элементов несколько, возвращается первый из них.

In [ ]:
words = ["banana", "kiwi", "apple", "fig", "cherry"]

print(max(words, key=lambda word: len(word)))  # banana — при равных ключах берется первый элемент
print(min(words, key=lambda word: len(word)))  # fig

In [ ]:
most_expensive = max(products, key=lambda product: product["price"])
cheapest = min(products, key=lambda product: product["price"])
largest_cost = max(products, key=lambda product: product["price"] * product["amount"])

print(most_expensive["label"])  # laptop
print(cheapest["label"])        # cable
print(largest_cost["label"])    # monitor

Поиск максимума среди пар словаря и поиск элемента, ближайшего к заданному
значению (lambda-функция обращается к внешнему имени `target`):

In [ ]:
best_name, best_score = max(scores.items(), key=lambda pair: pair[1])
print(f"{best_name}: {best_score}")  # Boris: 95

numbers = [3, 8, 14, 11, 20]
target = 10

print(min(numbers, key=lambda number: abs(number - target)))  # 11

Для пустой коллекции `max` и `min` вызывают `ValueError`. Параметр `default`
задает результат, который возвращается вместо ошибки:

In [ ]:
print(max([], key=lambda word: len(word), default="no words"))  # no words

In [ ]:
max([], key=lambda word: len(word))

### Рекомендации по использованию lambda

- `lambda` уместна для короткого выражения, которое используется один раз:
  ключ сортировки, критерий поиска максимума или минимума;
- если в качестве ключа подходит готовая функция (`len`, `abs`, `str.lower`),
  ее следует передавать напрямую;
- если выражение не помещается в одну короткую строку или используется
  повторно, следует определить функцию через `def` с понятным именем,
  аннотациями типов и docstring.

## 2. Паттерн проектирования «Стратегия»

### Паттерны проектирования

**Паттерн проектирования** — описание типового решения часто возникающей
задачи организации кода. Классическим источником считается книга
*Design Patterns* (Gamma, Helm, Johnson, Vlissides, 1994), в которой описаны
23 паттерна. Паттерны сформулированы независимо от языка программирования,
поэтому их конкретная реализация зависит от возможностей языка.

Рассмотрим паттерн **«Стратегия»** (*Strategy*) и покажем, что в Python его
можно реализовать без классов.

### Классическая реализация

Исходное определение паттерна: *определить семейство алгоритмов,
инкапсулировать каждый из них и сделать их взаимозаменяемыми.* Стратегия
позволяет менять алгоритм независимо от кода, который им пользуется.

![Структура паттерна Стратегия](images/strategy.svg)

- `Context` — объект, который использует алгоритм. Он хранит ссылку на
  стратегию и делегирует ей выполнение работы;
- `AbstractStrategy` — общий интерфейс всех алгоритмов;
- `ConcreteStrategyA`, `ConcreteStrategyB`, `ConcreteStrategyC` — конкретные
  алгоритмы, реализующие этот интерфейс.

В языках, где функция не является полноценным объектом, для каждого
алгоритма приходится объявлять отдельный класс.

### Стратегия на функциях в Python

В Python функция — обычный объект: ее можно передать в другую функцию,
сохранить в коллекции и вызвать позже (см. конспект о функциях как объектах
высшего порядка). Поэтому:

- интерфейс `AbstractStrategy` заменяется сигнатурой функции (в аннотациях
  типов — `Callable`);
- каждая конкретная стратегия — обычная функция с этой сигнатурой;
- контекст получает стратегию в виде аргумента и вызывает ее.

Классы `ConcreteStrategy` и абстрактный базовый класс не нужны.

### Пример: скидки в интернет-магазине

Интернет-магазин предоставляет покупателям скидки. Данные хранятся в обычных
словарях:

- **позиция заказа** (`item`): название `label`, цена за единицу `price` и
  количество единиц `amount`;
- **покупатель** (`customer`): имя `username` и баллы лояльности
  `loyalty_points`;
- **заказ** (`order`): покупатель `customer` и список позиций `items`.

In [ ]:
item = {"label": "notebook", "price": 150.0, "amount": 3}
customer = {"username": "alice", "loyalty_points": 1200}

order = {
    "customer": customer,
    "items": [
        item,
        {"label": "pen", "price": 20.0, "amount": 10},
    ],
}

print(order["customer"]["loyalty_points"])  # 1200
print(order["items"][1]["label"])           # pen

Магазин использует три правила начисления скидки. Скидка — это **денежная
сумма**, на которую уменьшается цена заказа:

1. **Скидка за лояльность.** Если у покупателя не менее 1000 баллов
   лояльности, скидка составляет 5% от суммы заказа.
2. **Скидка за количество единиц позиции.** Для каждой позиции, которой
   заказано не менее 20 единиц, скидка составляет 10% от стоимости этой
   позиции (`price * amount`). Скидки по разным позициям суммируются.
3. **Скидка за разнообразие заказа.** Если в заказе не менее 10 уникальных
   позиций (с разными значениями `label`), скидка составляет 7% от суммы
   заказа.

К заказу применяется **только одна** скидка — та, которая дает покупателю
наибольшую выгоду.

Каждое правило оформляется как отдельная стратегия: функция, которая
принимает заказ и возвращает размер скидки. Заказ выступает контекстом: он
предоставляет данные для расчета. Выбор лучшей стратегии сводится к
применению `max` с параметром `key`.

![Стратегии начисления скидки](images/strategy_discount.svg)

Для решения подготовлены вспомогательные функции: создание позиции и заказа,
расчет суммы заказа без скидок. Вещественные числа в проверках сравниваются
функцией `math.isclose`.

In [ ]:
import math
from typing import Callable


def create_item(label: str, price: float, amount: int = 1) -> dict:
    """Create an order item."""
    return {"label": label, "price": price, "amount": amount}


def create_order(loyalty_points: int, items: list[dict]) -> dict:
    """Create an order of a customer with the given loyalty points."""
    return {
        "customer": {"username": "tester", "loyalty_points": loyalty_points},
        "items": items,
    }


def get_order_price(order: dict) -> float:
    """Return the total price of the order without discounts."""
    return sum(item["price"] * item["amount"] for item in order["items"])

In [ ]:
print(get_order_price(order))        # 650.0
print(create_item("pen", 20.0, 10))  # {'label': 'pen', 'price': 20.0, 'amount': 10}
print(create_item("pen", 20.0))      # {'label': 'pen', 'price': 20.0, 'amount': 1}

## 3. Практическая задача: скидка за лояльность

**Условие.**

Реализуйте функцию-стратегию `get_loyalty_discount`. Она принимает заказ
`order` и возвращает размер скидки за лояльность: 5% от суммы заказа, если у
покупателя не менее 1000 баллов лояльности, иначе `0.0`.

Например, для заказа на сумму 200 при 1000 баллах лояльности результат
равен `10.0`, а при 999 баллах — `0.0`.

In [ ]:
def get_loyalty_discount(order: dict) -> float:
    # ваш код
    return 0.0

### Проверка

In [ ]:
test_cases = [
    (create_order(0, [create_item("pen", 100.0, 2)]), 0.0),
    (create_order(999, [create_item("pen", 100.0, 2)]), 0.0),
    (create_order(1000, [create_item("pen", 100.0, 2)]), 10.0),
    (create_order(5000, [create_item("pen", 100.0, 2), create_item("book", 50.0)]), 12.5),
    (create_order(1000, []), 0.0),
]

for case_number, (test_order, discount_expected) in enumerate(test_cases):
    discount = get_loyalty_discount(test_order)

    assert math.isclose(discount, discount_expected), f"case {case_number}: {discount} != {discount_expected}"

print("All tests passed")

## 4. Практическая задача: скидка за количество единиц позиции

**Условие.**

Реализуйте функцию-стратегию `get_item_amount_discount`. Она принимает заказ
`order` и возвращает суммарный размер скидки по позициям. Скидка считается
отдельно для каждой позиции заказа:

- если заказано не менее 20 единиц позиции (`amount >= 20`), скидка по ней
  составляет 10% от стоимости позиции (`price * amount`);
- иначе скидка по позиции равна нулю.

Результат функции — сумма скидок по всем позициям. Если ни по одной позиции
скидка не положена, результат равен `0.0`.

**Пример.** Заказ состоит из трех позиций:

| `label` | `price` | `amount` | `amount >= 20` | стоимость позиции | скидка по позиции |
|---|---|---|---|---|---|
| `"pen"` | `10.0` | `20` | да | `10.0 * 20 = 200.0` | `200.0 * 0.1 = 20.0` |
| `"book"` | `5.0` | `30` | да | `5.0 * 30 = 150.0` | `150.0 * 0.1 = 15.0` |
| `"laptop"` | `100.0` | `1` | нет | `100.0 * 1 = 100.0` | `0.0` |

Суммарная скидка: `20.0 + 15.0 + 0.0 = 35.0`.

In [ ]:
def get_item_amount_discount(order: dict) -> float:
    # ваш код
    return 0.0

### Проверка

In [ ]:
test_cases = [
    (create_order(0, []), 0.0),
    (create_order(0, [create_item("pen", 10.0, 19)]), 0.0),
    (create_order(0, [create_item("pen", 10.0, 20)]), 20.0),
    (
        create_order(
            0,
            [
                create_item("pen", 10.0, 20),
                create_item("book", 5.0, 30),
                create_item("laptop", 100.0),
            ],
        ),
        35.0,
    ),
    (create_order(5000, [create_item("pen", 10.0, 100)]), 100.0),
]

for case_number, (test_order, discount_expected) in enumerate(test_cases):
    discount = get_item_amount_discount(test_order)

    assert math.isclose(discount, discount_expected), f"case {case_number}: {discount} != {discount_expected}"

print("All tests passed")

## 5. Практическая задача: скидка за разнообразие заказа

**Условие.**

Реализуйте функцию-стратегию `get_general_amount_discount`. Она принимает
заказ `order` и возвращает 7% от суммы заказа, если в заказе не менее 10
уникальных позиций (с разными значениями `label`), иначе `0.0`.

Например, для заказа из 10 позиций с разными названиями общей стоимостью
1000 результат равен `70.0`, а для 10 позиций с одним и тем же названием —
`0.0`.

In [ ]:
def get_general_amount_discount(order: dict) -> float:
    # ваш код
    return 0.0

### Проверка

In [ ]:
unique_items = [create_item(f"item_{number}", 100.0) for number in range(10)]

test_cases = [
    (create_order(0, []), 0.0),
    (create_order(0, unique_items[:9]), 0.0),
    (create_order(0, unique_items), 70.0),
    (create_order(0, unique_items[:9] + [create_item("item_0", 100.0)]), 0.0),
    (create_order(0, [create_item("pen", 100.0) for _ in range(10)]), 0.0),
    (create_order(0, [create_item(f"item_{number}", 50.0, 2) for number in range(12)]), 84.0),
]

for case_number, (test_order, discount_expected) in enumerate(test_cases):
    discount = get_general_amount_discount(test_order)

    assert math.isclose(discount, discount_expected), f"case {case_number}: {discount} != {discount_expected}"

print("All tests passed")

## 6. Практическая задача: выбор лучшей скидки

**Условие.**

Реализуйте функцию `get_discounted_price`, которая принимает заказ `order` и
список функций-стратегий `discount_functions`. Каждая стратегия принимает
заказ и возвращает размер скидки. Функция должна выбрать стратегию, которая
дает наибольшую скидку, и вернуть цену заказа с учетом этой скидки. Если
список стратегий пуст, возвращается цена заказа без скидки.

Например, для заказа на сумму 200 и стратегий, возвращающих скидки `10.0` и
`25.0`, результат равен `175.0`.

In [ ]:
def get_discounted_price(
    order: dict,
    discount_functions: list[Callable[[dict], float]],
) -> float:
    # ваш код
    return get_order_price(order)

### Проверка

In [ ]:
plain_order = create_order(0, [create_item("pen", 100.0, 2)])

assert math.isclose(get_discounted_price(plain_order, []), 200.0)
assert math.isclose(get_discounted_price(plain_order, [lambda current_order: 10.0]), 190.0)
assert math.isclose(
    get_discounted_price(
        plain_order,
        [lambda current_order: 10.0, lambda current_order: 25.0, lambda current_order: 0.0],
    ),
    175.0,
)
assert math.isclose(
    get_discounted_price(plain_order, [lambda current_order: get_order_price(current_order) / 2]),
    100.0,
)

print("All tests passed")

Проверим функцию на реальных стратегиях. Для этого должны быть реализованы
задачи 3–5.

In [ ]:
all_discounts = [
    get_loyalty_discount,
    get_item_amount_discount,
    get_general_amount_discount,
]

loyalty_order = create_order(1000, [create_item("pen", 100.0, 2)])
item_amount_order = create_order(1000, [create_item("pen", 10.0, 20)])
general_order = create_order(0, [create_item(f"item_{number}", 50.0, 2) for number in range(12)])
mixed_order = create_order(1000, [create_item(f"item_{number}", 50.0, 2) for number in range(12)])

assert math.isclose(get_discounted_price(loyalty_order, all_discounts), 190.0)
assert math.isclose(get_discounted_price(item_amount_order, all_discounts), 180.0)
assert math.isclose(get_discounted_price(general_order, all_discounts), 1116.0)
assert math.isclose(get_discounted_price(mixed_order, all_discounts), 1116.0)
assert math.isclose(get_discounted_price(plain_order, all_discounts), 200.0)

print("All tests passed")